# 🎬 Montana State University M090: Recordly Motion Engine (Google Colab GPU Edition)

교수들과 학생들이 극찬한 **카메라 스프링 줌·패닝, 좌·우 패널 포커스, 턴 트래킹 네온 링, 스튜디오 윈도우 프레임**을 그대로 탑재한 **공식 Recordly 비디오 연출 엔진**을 **Google 무료 T4 GPU** 환경에서 초고속으로 구동하는 전용 Colab 노트북입니다.

---
### 🌟 이번 실행 목표: UNIT 3 (Lecture 31 ~ 45) 초고속 분산 렌더링
- **내 PC:** UNIT 2 (Lecture 16 ~ 30) 렌더링 전담
- **Google Colab (이 노트북):** UNIT 3 (Lecture 31 ~ 45) 무료 T4 GPU 전담
- **결과:** PC 부담 없이 2배 빠른 속도로 전 커리큘럼 완성!

## 1단계: 무료 GPU 활성화 및 Linux 가상 디스플레이(Xvfb) 환경 구축
상단 메뉴의 **[런타임] → [런타임 유형 변경]**에서 하드웨어 가속기가 **T4 GPU**로 지정되어 있는지 확인 후 실행하세요.

In [ ]:
# 1. GPU 하드웨어 점검
!nvidia-smi

# 2. 리눅스 가상 디스플레이(Xvfb) 및 브라우저 의존성 설치
!apt-get update -qq && apt-get install -y -qq xvfb libxkbcommon-x11-0 libgbm-dev

# 3. 파이썬 라이브러리 및 Playwright Chromium 설치
!pip install -q edge-tts imageio-ffmpeg playwright
!playwright install chromium
!playwright install-deps

## 2단계: 가상 디스플레이(Xvfb) 가동 & 최신 저장소 클론

In [ ]:
import os
import subprocess

# 1920x1080 60fps 가상 디스플레이 가동
subprocess.Popen(["Xvfb", ":99", "-screen", "0", "1920x1080x24"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
os.environ["DISPLAY"] = ":99"
print("🖥️ Linux 가상 디스플레이 (:99, 1920x1080) 시작 완료!")

# GitHub 프로젝트 최신 코드 클론
%cd /content
if os.path.exists('/content/montana-lecture-app'):
    !rm -rf /content/montana-lecture-app
!git clone https://github.com/marylatte11-svg/montana-lecture-app.git /content/montana-lecture-app
%cd /content/montana-lecture-app
!pwd

## 3단계 (선택): 구글 드라이브 마운트
완성된 강의 마스터 영상을 내 구글 드라이브에 자동 보관하려면 실행하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/MSU_Recordly_Lectures_Unit3'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"☁️ 구글 드라이브 저장 경로: {DRIVE_DIR}")

## 4단계: UNIT 3 (Lecture 31 ~ 45) Recordly 일괄 렌더링 시작
무료 T4 GPU의 `h264_nvenc` 하드웨어 가속으로 Lecture 31부터 45까지 차례로 렌더링됩니다.

In [ ]:
# UNIT 3 전체 (Lecture 31 ~ 45) 자동 일괄 렌더링
!python scripts/recordly_engine/render_unit1_batch.py --start 31 --end 45

## 5단계: 완성된 비디오를 구글 드라이브로 일괄 백업

In [ ]:
import glob
import shutil

master_videos = glob.glob("/content/montana-lecture-app/Montana_State_Univ/recordly_videos/*/*_Full_Master.mp4")
print(f"🎬 총 {len(master_videos)}개의 풀 마스터 비디오가 완성되었습니다.")

if os.path.exists('/content/drive/MyDrive'):
    for v in sorted(master_videos):
        dest = os.path.join(DRIVE_DIR, os.path.basename(v))
        shutil.copy2(v, dest)
        sz = os.path.getsize(dest) / (1024 * 1024)
        print(f"  ☁️ 백업 완료: {os.path.basename(v)} ({sz:.2f} MB)")
    print("🎉 모든 UNIT 3 비디오가 Google Drive에 안전하게 저장되었습니다!")
else:
    print("ℹ️ 구글 드라이브가 마운트되지 않았습니다. 3단계를 실행하면 자동 백업됩니다.")